In [3]:
# [1]: Importar librerías necesarias
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col
from pyspark.sql.types import StringType

# [2]: Crear sesión de Spark
spark = SparkSession.builder \
    .appName("KafkaWordCount") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# [3]: Leer stream desde Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "actividad-topic") \
    .load()

In [4]:
kafka_df_2 = kafka_df.selectExpr("CAST(value as STRING)")

# Procesar el conteo de palabras
word_count_df = kafka_df_2 \
    .select(explode(split(col("value"), " ")).alias("word")) \
    .groupBy("word") \
    .count()

# Escribir el stream a memoria
query = word_count_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("word_counts_table") \
    .start()

In [5]:
import time 
while True:
    try:
        from IPython.display import clear_output
        clear_output(wait=True)

        print("Consultando resultados en vivo...")
        results_df = spark.sql("SELECT * FROM word_counts_table ORDER BY count DESC")
        results_df.show()

        time.sleep(5)

    except KeyboardInterrupt:
        print("Consulta en vivo detenida")
        break

Consultando resultados en vivo...
+-------------+-----+
|         word|count|
+-------------+-----+
|        datos|  406|
|           de|  279|
|         real|  277|
|           en|  277|
|       tiempo|  277|
|        spark|  275|
|          los|  272|
|   desarrollo|  145|
|      nuestro|  145|
|       docker|  145|
|   simplifica|  145|
|      entorno|  145|
|procesamiento|  142|
|            y|  138|
| herramientas|  138|
|          muy|  138|
|     potentes|  138|
|        kafka|  138|
|          son|  138|
|      procesa|  137|
+-------------+-----+
only showing top 20 rows

Consulta en vivo detenida
